# Week 3 Thursday Exercise 2 — EDD Decision Tree Solution

This notebook solves the EDD routing exercise using a simple decision tree.


In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [2]:
df = pd.read_csv('week3_thursday_edd_synthetic_data.csv')
df


,customer_id,customer_name,customer_type,geography,crr_score_0_100,adverse_media,pep,sanctions,ownership_complexity,cross_border_activity,cash_intensity,expected_activity_mismatch,transaction_spike,sudden_profile_change,current_review_outcome
0,H001,Atlas Import Group,Business,Cross-border,78,1,0,0,1,1,0,0,1,1,EDD
1,H002,Bright Side Retail,Retail,Domestic,42,0,0,0,0,0,0,0,0,0,Standard
2,H003,Cedar Family Office,Private wealth,Cross-border,82,0,1,0,1,0,1,0,0,0,EDD
3,H004,Delta Community Health,Business,Domestic,46,0,0,0,0,0,0,0,0,0,Standard
4,H005,Evergreen Payments LLC,Business,Cross-border,91,1,1,1,1,1,1,1,1,1,EDD
5,H006,Frontier Logistics,Business,Domestic,67,0,0,0,1,1,0,0,1,0,EDD
6,H007,Granite Travel Co,Business,Cross-border,58,0,0,0,1,0,1,0,0,1,EDD
7,H008,Harbor Local Markets,Retail,Domestic,35,0,0,0,0,0,0,0,0,0,Standard


## Define the target
EDD is triggered when the score is high or at least one trigger bucket is positive.


In [3]:
trigger_cols = ['adverse_media','pep','sanctions','ownership_complexity','cross_border_activity','cash_intensity','expected_activity_mismatch','transaction_spike','sudden_profile_change']
df['edd_target'] = ((df['crr_score_0_100'] >= 75) | (df[trigger_cols].sum(axis=1) >= 1)).astype(int)
df[['customer_id','customer_name','crr_score_0_100','edd_target']]


,customer_id,customer_name,crr_score_0_100,edd_target
0,H001,Atlas Import Group,78,1
1,H002,Bright Side Retail,42,0
2,H003,Cedar Family Office,82,1
3,H004,Delta Community Health,46,0
4,H005,Evergreen Payments LLC,91,1
5,H006,Frontier Logistics,67,1
6,H007,Granite Travel Co,58,1
7,H008,Harbor Local Markets,35,0


## Train the decision tree


In [4]:
features = ['crr_score_0_100'] + trigger_cols
X = df[features]
y = df['edd_target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
accuracy_score(y_test, pred)


1.0

## Evaluate the model


In [5]:
print(classification_report(y_test, pred, zero_division=0))
print(confusion_matrix(y_test, pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2

[[1 0]
 [0 1]]


## Decision rules


In [6]:
print(export_text(clf, feature_names=features))


|--- ownership_complexity <= 0.50
|   |--- class: 0
|--- ownership_complexity >  0.50
|   |--- class: 1



## Apply to all customers


In [7]:
df['predicted_edd'] = clf.predict(X)
df['predicted_label'] = df['predicted_edd'].map({0:'Standard',1:'EDD'})
df[['customer_id','customer_name','crr_score_0_100','edd_target','predicted_label']]


,customer_id,customer_name,crr_score_0_100,edd_target,predicted_label
0,H001,Atlas Import Group,78,1,EDD
1,H002,Bright Side Retail,42,0,Standard
2,H003,Cedar Family Office,82,1,EDD
3,H004,Delta Community Health,46,0,Standard
4,H005,Evergreen Payments LLC,91,1,EDD
5,H006,Frontier Logistics,67,1,EDD
6,H007,Granite Travel Co,58,1,EDD
7,H008,Harbor Local Markets,35,0,Standard


## Solution summary
The tree learns to route customers to EDD when the score is high or the trigger buckets are active.
